# imports, mounts, core + metrics import

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("KAGGLE_KEY")
secret_value_1 = user_secrets.get_secret("KAGGLE_USERNAME")

In [2]:
import sys
import os
import glob
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings(
    "ignore",
    message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated",
    category=FutureWarning,
)

try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _sec.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _sec.get_secret("KAGGLE_KEY")
    print("kaggle credentials loaded from secrets")
except Exception as exc:
    print("kaggle secrets not loaded:", exc)

def _import_from_glob(name):
    try:
        return __import__(name)
    except ModuleNotFoundError:
        hits = glob.glob(f"/kaggle/input/**/{name}.py", recursive=True)
        if not hits:
            raise ModuleNotFoundError(f"{name}.py not found under /kaggle/input")
        d = os.path.dirname(hits[0])
        if d not in sys.path:
            sys.path.insert(0, d)
        return __import__(name)

core = _import_from_glob("core")
metrics = _import_from_glob("metrics")

print("import-time CONFIG_HASH:", core.CONFIG_HASH)
print("metrics constants: ECE_BINS=%d BOOTSTRAP_N=%d BOOTSTRAP_SEED=%d" % (
    metrics.ECE_BINS, metrics.BOOTSTRAP_N, metrics.BOOTSTRAP_SEED))

REPORT_NOTES = []

kaggle credentials loaded from secrets
import-time CONFIG_HASH: 03fa342b535795c1
metrics constants: ECE_BINS=15 BOOTSTRAP_N=2000 BOOTSTRAP_SEED=13579


# pull store, stage frozen config, refresh, resolve paths, assert hash

In [3]:
pulled_ok = False
if core.DATAWISE_RESULTS_SLUG is not None:
    try:
        core.pull_results_store()
        pulled_ok = True
        print("pulled results store (latest) from", core.DATAWISE_RESULTS_SLUG)
    except Exception as exc:
        print("PULL FAILED:", exc)

frozen_target = core.PATHS["frozen_config"]
if not os.path.exists(frozen_target):
    hits = glob.glob("/kaggle/input/**/frozen_config.md", recursive=True)
    if not hits:
        raise FileNotFoundError("frozen_config.md not found; run 00 and mount the store")
    core._ensure(os.path.dirname(frozen_target))
    shutil.copy(hits[0], frozen_target)
    print("staged frozen_config.md from", hits[0])

h = core.refresh_config()
print("active CONFIG_HASH after refresh:", h)

def _resolve(name):
    local = os.path.join(core.PATHS["root"], name)
    if os.path.exists(local):
        return local
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if hits:
        return hits[0]
    return None

def _resolve_dir(dirname, sentinel):
    local = os.path.join(core.PATHS["root"], dirname)
    if os.path.isdir(local) and glob.glob(os.path.join(local, sentinel)):
        return local
    hits = glob.glob(f"/kaggle/input/**/{dirname}/{sentinel}", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    return local

pcam_ledger_path = _resolve(os.path.join("results", "runs_ledger.csv")) \
    or _resolve("runs_ledger.csv")
isic_ledger_path = _resolve(os.path.join("results", "isic_runs_ledger.csv")) \
    or _resolve("isic_runs_ledger.csv")
logits_dir = _resolve_dir("logits", "*_test.csv")
curves_dir = _resolve_dir("training_curves", "*.csv")

print("PCam ledger:", pcam_ledger_path)
print("ISIC ledger:", isic_ledger_path)
print("logits dir:", logits_dir)
print("curves dir:", curves_dir)

if pcam_ledger_path is None or not os.path.exists(pcam_ledger_path):
    raise FileNotFoundError("PCam ledger not found; cannot aggregate")

_ledger_hashes = pd.read_csv(pcam_ledger_path)["config_hash"].astype(str).unique()
print("ledger config_hash values:", list(_ledger_hashes))
assert len(_ledger_hashes) == 1, f"ledger has multiple config_hash values: {_ledger_hashes}"
assert _ledger_hashes[0] == core.CONFIG_HASH, (
    f"PROVENANCE MISMATCH: active {core.CONFIG_HASH} vs ledger {_ledger_hashes[0]}")
print("hash provenance confirmed:", core.CONFIG_HASH)

for d in ("results", "figures", "summaries"):
    core._ensure(os.path.join(core.PATHS["root"], d))
RESULTS = os.path.join(core.PATHS["root"], "results")
FIGURES = os.path.join(core.PATHS["root"], "figures")
SUMM = os.path.join(core.PATHS["root"], "summaries")

Dataset URL: https://www.kaggle.com/datasets/hrushisanap/datawise-results
pulled results store (latest) from hrushisanap/datawise-results
active CONFIG_HASH after refresh: 03fa342b535795c1
PCam ledger: /kaggle/working/results/runs_ledger.csv
ISIC ledger: /kaggle/working/results/isic_runs_ledger.csv
logits dir: /kaggle/working/logits
curves dir: /kaggle/working/training_curves
ledger config_hash values: ['03fa342b535795c1']
hash provenance confirmed: 03fa342b535795c1


# load and dedup the PCam ledger

In [4]:
L = pd.read_csv(pcam_ledger_path)

order = {"failed": 0, "success": 1}
L = (L.assign(_r=L["status"].map(order).fillna(-1))
       .sort_values("_r").drop_duplicates("run_id", keep="last").drop(columns="_r"))

n_success = int((L["status"] == "success").sum())
n_failed = int((L["status"] == "failed").sum())
print(f"deduped ledger: {len(L)} unique run_ids, {n_success} success, {n_failed} failed")

if n_failed:
    print("\nfailed run_ids:")
    for _, r in L[L["status"] == "failed"].iterrows():
        print(f"  {r['run_id']} (model={r['model']} n_min={r['n_min']} "
              f"n_maj={r['n_maj']} seed={r['data_seed']})")

print("\nper-protocol coverage (str.contains on protocol field):")
for pname in ("protocolA", "protocolB", "protocolC", "protocolD", "init_sensitivity"):
    m = L["protocol"].astype(str).str.contains(pname)
    s = int((m & (L["status"] == "success")).sum())
    print(f"  {pname}: {s} success rows")

EXPECTED_UNIQUE = 241
succ = L[L["status"] == "success"]
print(f"\nexpected unique successful runs: {EXPECTED_UNIQUE}; actual: {len(succ)}")
if len(succ) != EXPECTED_UNIQUE:
    print(f"  NOTE: count differs from {EXPECTED_UNIQUE}; investigate before trusting aggregates")

deduped ledger: 241 unique run_ids, 241 success, 0 failed

per-protocol coverage (str.contains on protocol field):
  protocolA: 63 success rows
  protocolB: 63 success rows
  protocolC: 63 success rows
  protocolD: 54 success rows
  init_sensitivity: 20 success rows

expected unique successful runs: 241; actual: 241


# build master_results.csv

In [5]:
LEDGER_IDENTITY = [
    "run_id", "condition_id", "protocol", "model", "n_min", "n_maj",
    "target_ratio", "actual_ratio", "data_seed", "init_seed", "loader_seed",
    "step_budget", "actual_steps", "effective_epochs", "exposures_per_image",
    "status", "config_hash",
]

def _load_logits(run_id, which):
    path = os.path.join(logits_dir, f"{run_id}_{which}.csv")
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    return df["y_true"].to_numpy().astype(int), df["logit_diff"].to_numpy().astype(np.float64)

rows = []
missing = []
for _, r in succ.iterrows():
    run_id = r["run_id"]
    test = _load_logits(run_id, "test")
    val = _load_logits(run_id, "val")
    if test is None or val is None:
        missing.append((run_id, "test" if test is None else "val"))
        continue
    y_test, d_test = test
    y_val, d_val = val
    n_min, n_maj = int(r["n_min"]), int(r["n_maj"])
    pi_train = n_min / (n_min + n_maj)
    m = metrics.compute_all_metrics(y_test, d_test, y_val, d_val, pi_train,
                                    with_bootstrap=False)
    row = {k: r[k] for k in LEDGER_IDENTITY}
    row["pi_train"] = pi_train
    row.update(m)
    rows.append(row)

if missing:
    print("runs skipped for missing logits:")
    for run_id, which in missing:
        print(f"  {run_id} (missing {which})")

master = pd.DataFrame(rows)
master_path = os.path.join(RESULTS, "master_results.csv")
master.to_csv(master_path, index=False)
print(f"\nwrote {master_path}")
print("shape:", master.shape)
print("columns:", list(master.columns))

expected_new = ["minority_ece", "majority_ece", "classwise_ece",
                "minority_ece_ts", "majority_ece_ts", "classwise_ece_ts",
                "minority_ece_prior", "majority_ece_prior", "classwise_ece_prior",
                "recall_pos_prior", "recall_neg_prior",
                "mean_conf_correct_min", "mean_conf_incorrect_min",
                "mean_conf_correct_maj", "mean_conf_incorrect_maj",
                "mean_entropy_correct_min", "mean_entropy_incorrect_min",
                "mean_entropy_correct_maj", "mean_entropy_incorrect_maj"]
absent = [c for c in expected_new if c not in master.columns]
assert not absent, f"patched metrics columns missing from master: {absent}"
print("all patched class-conditional columns present")

_mh = master["config_hash"].astype(str).unique()
assert len(_mh) == 1 and _mh[0] == core.CONFIG_HASH, f"hash mismatch in master: {_mh}"
print("config_hash in master confirmed:", _mh[0])

print("\nfirst rows:")
print(master.head(3).to_string())


wrote /kaggle/working/results/master_results.csv
shape: (241, 59)
columns: ['run_id', 'condition_id', 'protocol', 'model', 'n_min', 'n_maj', 'target_ratio', 'actual_ratio', 'data_seed', 'init_seed', 'loader_seed', 'step_budget', 'actual_steps', 'effective_epochs', 'exposures_per_image', 'status', 'config_hash', 'pi_train', 'bal_acc', 'recall_pos', 'recall_neg', 'auroc', 'brier', 'nll', 'ece', 'classwise_ece', 'minority_ece', 'majority_ece', 'mean_conf_correct', 'mean_conf_incorrect', 'mean_entropy_correct', 'mean_entropy_incorrect', 'delta_conf', 'aurc_overall', 'aurc_minority', 'aurc_majority', 'error_auroc', 'temperature', 'nll_ts', 'ece_ts', 'classwise_ece_ts', 'minority_ece_ts', 'majority_ece_ts', 'ece_prior', 'classwise_ece_prior', 'minority_ece_prior', 'majority_ece_prior', 'brier_prior', 'nll_prior', 'recall_pos_prior', 'recall_neg_prior', 'mean_conf_correct_min', 'mean_conf_incorrect_min', 'mean_entropy_correct_min', 'mean_entropy_incorrect_min', 'mean_conf_correct_maj', 'mean

# data-quality dashboard

In [6]:
dq = []
def _dq(line):
    dq.append(line)
    print(line)

_dq("# Notebook 07: data-quality dashboard")
_dq("")

# (i) the calibration-collapse detector: the patched metrics must no longer
# return identical ece / classwise_ece / minority_ece.
d_min = float((master["ece"] - master["minority_ece"]).abs().max())
d_cw = float((master["ece"] - master["classwise_ece"]).abs().max())
_dq(f"- max|ece - minority_ece| across runs: {d_min:.6f}")
_dq(f"- max|ece - classwise_ece| across runs: {d_cw:.6f}")
if d_min <= 0.0 or d_cw <= 0.0:
    raise RuntimeError(
        "calibration metrics are still identical (collapse not fixed): "
        f"max|ece-minority_ece|={d_min}, max|ece-classwise_ece|={d_cw}. "
        "The old metrics.py is likely still mounted. STOP.")
_dq("  calibration metrics are genuinely class-conditional (collapse fixed)")

n_logit_files = len(glob.glob(os.path.join(logits_dir, "*_test.csv"))) + \
                len(glob.glob(os.path.join(logits_dir, "*_val.csv")))
_dq(f"- logit files (test+val): {n_logit_files}; master rows x2 = {len(master) * 2}")

metric_cols = [c for c in master.columns if c not in LEDGER_IDENTITY + ["pi_train"]]
bad = []
for c in metric_cols:
    vals = pd.to_numeric(master[c], errors="coerce")
    nanmask = ~np.isfinite(vals)
    if nanmask.any():
        for rid in master.loc[nanmask, "run_id"]:
            bad.append((rid, c))
if bad:
    _dq(f"- NaN/inf found in {len(bad)} (run_id, column) pairs:")
    for rid, c in bad[:50]:
        _dq(f"    {rid}  {c}")
    if len(bad) > 50:
        _dq(f"    ... and {len(bad) - 50} more")
else:
    _dq("- NaN/inf: none in any metric column")

tb = master[(master["temperature"] <= 0.05 * 1.01) | (master["temperature"] >= 20.0 * 0.99)]
if len(tb):
    _dq(f"- temperature within 1% of a bound (possible degenerate TS fit): {len(tb)} runs")
    for _, r in tb.iterrows():
        _dq(f"    {r['run_id']}  T={r['temperature']:.4f}")
else:
    _dq("- temperature: no runs near the optimiser bounds")

short = master[master["actual_steps"] != master["step_budget"]]
_dq(f"- short runs (actual_steps != step_budget): {len(short)}")
for _, r in short.iterrows():
    _dq(f"    {r['run_id']}  actual={r['actual_steps']} budget={r['step_budget']}")

hashes = master["config_hash"].astype(str).unique()
_dq(f"- config_hash single-valued: {len(hashes) == 1} ({list(hashes)})")

# (ii) seed coverage, counting only the main three-seed ordinary runs per
# (protocol, model, condition); init_sensitivity rows are held separate.
_dq("- seed coverage per (protocol-tag, model, condition), expecting {21,22,23}:")
gaps = 0
for pname in ("protocolA", "protocolB", "protocolC", "protocolD"):
    sub = master[(master["protocol"].astype(str).str.contains(pname)) &
                 (~master["protocol"].astype(str).str.contains("init_sensitivity"))]
    for (model, cid), g in sub.groupby(["model", "condition_id"]):
        seeds = sorted(set(g["data_seed"].tolist()))
        if seeds != [21, 22, 23]:
            _dq(f"    GAP {pname} {model} {cid}: seeds {seeds}")
            gaps += 1
if gaps == 0:
    _dq("    all (protocol, model, condition) groups have seeds {21,22,23}")

n_main = int((~master["protocol"].astype(str).str.contains("init_sensitivity")).sum())
n_init = int(master["protocol"].astype(str).str.contains("init_sensitivity").sum())
_dq(f"- PCam main successful runs (non-init): {n_main}")
_dq(f"- init-sensitivity runs (held separate): {n_init}")
_dq("  (expected ~241 main; print actual rather than asserting a remembered number)")

dq_path = os.path.join(SUMM, "07_data_quality.md")
with open(dq_path, "w") as f:
    f.write("\n".join(dq) + "\n")
print("\nwrote", dq_path)

# Notebook 07: data-quality dashboard

- max|ece - minority_ece| across runs: 0.449413
- max|ece - classwise_ece| across runs: 0.067308
  calibration metrics are genuinely class-conditional (collapse fixed)
- logit files (test+val): 482; master rows x2 = 482
- NaN/inf: none in any metric column
- temperature: no runs near the optimiser bounds
- short runs (actual_steps != step_budget): 0
- config_hash single-valued: True (['03fa342b535795c1'])
- seed coverage per (protocol-tag, model, condition), expecting {21,22,23}:
    GAP protocolA cnn m2000_M2000: seeds [22, 23]
    GAP protocolA logistic m2000_M2000: seeds [22, 23]
    GAP protocolC cnn m1000_M1000: seeds [22, 23]
    GAP protocolC logistic m1000_M1000: seeds [22, 23]
- PCam main successful runs (non-init): 221
- init-sensitivity runs (held separate): 20
  (expected ~241 main; print actual rather than asserting a remembered number)

wrote /kaggle/working/summaries/07_data_quality.md


# shared-condition consistency check

In [7]:
shared_cid = core.make_condition_id(32000, 32000)
print("shared condition_id:", shared_cid)

shared = master[master["condition_id"] == shared_cid]
print(f"rows at shared condition: {len(shared)} (expect 9: 3 models x 3 seeds)")

key_metrics = ["bal_acc", "auroc", "brier", "minority_ece", "aurc_minority"]
print("\nrun_id and key metrics at (32000,32000):")
for _, r in shared.sort_values(["model", "data_seed"]).iterrows():
    print(f"  {r['run_id']}  protocol={r['protocol']}")
    print("    " + "  ".join(f"{k}={r[k]:.4f}" for k in key_metrics))

for model in core.MODEL_NAMES:
    for s in core.SEEDS:
        rid = core.make_run_id(model, 32000, 32000, s, s, s)
        g = shared[shared["run_id"] == rid]
        assert len(g) == 1, f"shared run {rid} not unique in master ({len(g)} rows)"
        assert "protocolA" in str(g.iloc[0]["protocol"]), f"{rid} missing protocolA tag"

print("\nconsistency check PASS: each shared point is one run_id reused across A/B/D")
REPORT_NOTES.append(("consistency",
    f"The (32000, 32000) point resolves to a single run_id per model-seed across "
    f"Protocols A, B and D ({len(shared)} rows), so its metrics are identical by "
    f"construction wherever a protocol references it."))

shared condition_id: m32000_M32000
rows at shared condition: 9 (expect 9: 3 models x 3 seeds)

run_id and key metrics at (32000,32000):
  cnn__m32000_M32000__d21_i21_l21  protocol=protocolA|protocolB|protocolD
    bal_acc=0.7829  auroc=0.8804  brier=0.1513  minority_ece=0.1538  aurc_minority=0.1592
  cnn__m32000_M32000__d22_i22_l22  protocol=protocolA|protocolB|protocolD
    bal_acc=0.7763  auroc=0.8865  brier=0.1520  minority_ece=0.1569  aurc_minority=0.1697
  cnn__m32000_M32000__d23_i23_l23  protocol=protocolA|protocolB|protocolD
    bal_acc=0.7869  auroc=0.8902  brier=0.1455  minority_ece=0.1323  aurc_minority=0.1610
  logistic__m32000_M32000__d21_i21_l21  protocol=protocolA|protocolB|protocolD
    bal_acc=0.6693  auroc=0.7184  brier=0.2224  minority_ece=0.1351  aurc_minority=0.3441
  logistic__m32000_M32000__d22_i22_l22  protocol=protocolA|protocolB|protocolD
    bal_acc=0.6654  auroc=0.7155  brier=0.2234  minority_ece=0.1402  aurc_minority=0.3477
  logistic__m32000_M32000__d23_i23

# seed aggregation helper

In [8]:
MAIN_SEEDS = [21, 22, 23]

def agg_metric(df, metric):
    """Mean, std and min-max of a metric across the three main seeds, per (model, condition)."""
    out = []
    for (model, cid), g in df.groupby(["model", "condition_id"]):
        g3 = g[g["data_seed"].isin(MAIN_SEEDS)]
        if len(g3) == 0:
            continue
        v = pd.to_numeric(g3[metric], errors="coerce").to_numpy()
        v = v[np.isfinite(v)]
        if len(v) == 0:
            continue
        out.append({
            "model": model, "condition_id": cid,
            "n_min": int(g3["n_min"].iloc[0]), "n_maj": int(g3["n_maj"].iloc[0]),
            "actual_ratio": float(g3["actual_ratio"].iloc[0]),
            "n_seeds": len(v),
            "mean": float(v.mean()), "std": float(v.std(ddof=0)),
            "min": float(v.min()), "max": float(v.max()),
        })
    return pd.DataFrame(out)

def protocol_rows(pname):
    return master[(master["protocol"].astype(str).str.contains(pname)) &
                  (master["status"] == "success")]

full3 = 0
for pname in ("protocolA", "protocolB", "protocolC", "protocolD"):
    a = agg_metric(protocol_rows(pname), "minority_ece")
    full3 += int((a["n_seeds"] == 3).sum())
print(f"(model, condition) groups with full 3 seeds (across protocols, minority_ece): {full3}")

HEADLINE = ["bal_acc", "auroc", "brier", "minority_ece", "aurc_minority"]
MODEL_STYLE = {"logistic": "o-", "cnn": "s-", "resnet18": "^-"}

def _savefig(fig, stem):
    png = os.path.join(FIGURES, stem + ".png")
    svg = os.path.join(FIGURES, stem + ".svg")
    fig.savefig(png, dpi=600, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {png} and .svg")

def _dump_df(df, label):
    print(f"\n=== {label} ===")
    print(df.round(4).to_string(index=False))

(model, condition) groups with full 3 seeds (across protocols, minority_ece): 81


# Figure 1, plane schematic

In [9]:
PROT_CONDS = {
    "A": [(s, s) for s in core.PROTOCOL_A_SIZES],
    "B": [(int(round(core.N_F / (1 + r))), core.N_F - int(round(core.N_F / (1 + r))))
          for r in core.RATIOS],
    "C": [(1000, 1000 * r) for r in core.RATIOS],
    "D": [(m, 32000) for m in [1000, 2000, 4000, 8000, 16000, 32000]],
}

fig, ax = plt.subplots(figsize=(6, 6))
colours = {"A": "C0", "B": "C1", "C": "C2", "D": "C3"}
rows_dump = []
for p, conds in PROT_CONDS.items():
    xs = [c[0] for c in conds]
    ys = [c[1] for c in conds]
    ax.plot(xs, ys, colours[p] + "o-", label=f"Protocol {p}", alpha=0.8)
    for (nmin, nmaj) in conds:
        rows_dump.append({"protocol": p, "n_min": nmin, "n_maj": nmaj})
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("minority count n_min"); ax.set_ylabel("majority count n_maj")
ax.set_title("Figure 1: the four protocols in the (n_min, n_maj) plane")
ax.legend()
_savefig(fig, "figure_01_plane_schematic")

dump = pd.DataFrame(rows_dump)
dump.to_csv(os.path.join(FIGURES, "figure_01_plane_schematic.csv"), index=False)
print("\n[Figure 1] every condition (n_min, n_maj) per protocol:")
print(dump.to_string(index=False))

saved /kaggle/working/figures/figure_01_plane_schematic.png and .svg

[Figure 1] every condition (n_min, n_maj) per protocol:
protocol  n_min  n_maj
       A   2000   2000
       A   4000   4000
       A   8000   8000
       A  16000  16000
       A  32000  32000
       A  64000  64000
       A 128000 128000
       B  32000  32000
       B  21333  42667
       B  10667  53333
       B   5818  58182
       B   2462  61538
       B   1255  62745
       B    634  63366
       C   1000   1000
       C   1000   2000
       C   1000   5000
       C   1000  10000
       C   1000  25000
       C   1000  50000
       C   1000 100000
       D   1000  32000
       D   2000  32000
       D   4000  32000
       D   8000  32000
       D  16000  32000
       D  32000  32000


# Figure 2, Protocol A metrics vs total N

In [10]:
A = protocol_rows("protocolA").copy()
A["total_N"] = A["n_min"] + A["n_maj"]

fig, axes = plt.subplots(1, len(HEADLINE), figsize=(4 * len(HEADLINE), 4))
all_rows = []
for ax, metric in zip(axes, HEADLINE):
    a = agg_metric(A, metric)
    a["total_N"] = a["n_min"] + a["n_maj"]
    for model in core.MODEL_NAMES:
        am = a[a["model"] == model].sort_values("total_N")
        ax.errorbar(am["total_N"], am["mean"], yerr=am["std"],
                    fmt=MODEL_STYLE[model], capsize=3, label=model)
        for _, rr in am.iterrows():
            all_rows.append({"metric": metric, "model": model, "total_N": rr["total_N"],
                             "mean": rr["mean"], "std": rr["std"],
                             "min": rr["min"], "max": rr["max"]})
    ax.set_xscale("log"); ax.set_xlabel("total N"); ax.set_title(metric)
    if metric == HEADLINE[0]:
        ax.legend(fontsize=7)
fig.suptitle("Figure 2: Protocol A (balanced quantity) vs total N")
_savefig(fig, "figure_02_protocolA_vs_N")

dump = pd.DataFrame(all_rows)
dump.to_csv(os.path.join(FIGURES, "figure_02_protocolA_vs_N.csv"), index=False)
_dump_df(dump, "Figure 2: Protocol A, metric vs total N (mean/std/min/max over 3 seeds)")
REPORT_NOTES.append(("protocolA",
    "Protocol A (balanced, varying total N): see figure_02 CSV for per-model trends "
    "in bal_acc, auroc, brier, minority_ece and aurc_minority as total N grows."))

saved /kaggle/working/figures/figure_02_protocolA_vs_N.png and .svg

=== Figure 2: Protocol A, metric vs total N (mean/std/min/max over 3 seeds) ===
       metric    model  total_N   mean    std    min    max
      bal_acc logistic     4000 0.6186 0.0043 0.6126 0.6220
      bal_acc logistic     8000 0.6361 0.0017 0.6345 0.6385
      bal_acc logistic    16000 0.6508 0.0019 0.6483 0.6528
      bal_acc logistic    32000 0.6595 0.0006 0.6588 0.6602
      bal_acc logistic    64000 0.6662 0.0023 0.6638 0.6693
      bal_acc logistic   128000 0.6695 0.0025 0.6660 0.6715
      bal_acc logistic   256000 0.6663 0.0013 0.6645 0.6675
      bal_acc      cnn     4000 0.7835 0.0038 0.7781 0.7866
      bal_acc      cnn     8000 0.7775 0.0049 0.7722 0.7840
      bal_acc      cnn    16000 0.7830 0.0025 0.7799 0.7860
      bal_acc      cnn    32000 0.7795 0.0035 0.7751 0.7836
      bal_acc      cnn    64000 0.7820 0.0044 0.7763 0.7869
      bal_acc      cnn   128000 0.7859 0.0113 0.7767 0.8019
      bal_a

# Figure 3, Protocol B vs actual ratio

In [11]:
B = protocol_rows("protocolB").copy()

fig, axes = plt.subplots(1, len(HEADLINE), figsize=(4 * len(HEADLINE), 4))
all_rows = []
for ax, metric in zip(axes, HEADLINE):
    a = agg_metric(B, metric)
    for model in core.MODEL_NAMES:
        am = a[a["model"] == model].sort_values("actual_ratio")
        ax.errorbar(am["actual_ratio"], am["mean"], yerr=am["std"],
                    fmt=MODEL_STYLE[model], capsize=3, label=model)
        for _, rr in am.iterrows():
            all_rows.append({"metric": metric, "model": model,
                             "actual_ratio": rr["actual_ratio"], "mean": rr["mean"],
                             "std": rr["std"], "min": rr["min"], "max": rr["max"]})
    ax.set_xscale("log"); ax.set_xlabel("actual ratio n_maj/n_min"); ax.set_title(metric)
    if metric == HEADLINE[0]:
        ax.legend(fontsize=7)
fig.suptitle("Figure 3: Protocol B (fixed total) vs actual ratio")
_savefig(fig, "figure_03_protocolB_vs_ratio")

dump = pd.DataFrame(all_rows)
dump.to_csv(os.path.join(FIGURES, "figure_03_protocolB_vs_ratio.csv"), index=False)
_dump_df(dump, "Figure 3: Protocol B, metric vs actual ratio (mean/std/min/max)")
print("\nnote: target r=100 lands at actual ~99.95 by the pinned int(round(N_F/(1+r))) formula")
REPORT_NOTES.append(("protocolB",
    "Protocol B (fixed total N_F=64000, varying ratio): the r=100 condition sits at "
    "actual ratio ~99.95 by the pinned count formula. See figure_03 CSV."))

saved /kaggle/working/figures/figure_03_protocolB_vs_ratio.png and .svg

=== Figure 3: Protocol B, metric vs actual ratio (mean/std/min/max) ===
       metric    model  actual_ratio   mean    std    min    max
      bal_acc logistic        1.0000 0.6662 0.0023 0.6638 0.6693
      bal_acc logistic        2.0000 0.6178 0.0007 0.6171 0.6187
      bal_acc logistic        4.9998 0.5740 0.0026 0.5721 0.5778
      bal_acc logistic       10.0003 0.5564 0.0008 0.5553 0.5572
      bal_acc logistic       24.9951 0.5443 0.0022 0.5415 0.5467
      bal_acc logistic       49.9960 0.5382 0.0020 0.5354 0.5401
      bal_acc logistic       99.9464 0.5352 0.0003 0.5350 0.5356
      bal_acc      cnn        1.0000 0.7820 0.0044 0.7763 0.7869
      bal_acc      cnn        2.0000 0.7562 0.0045 0.7498 0.7597
      bal_acc      cnn        4.9998 0.6927 0.0099 0.6815 0.7055
      bal_acc      cnn       10.0003 0.6377 0.0015 0.6356 0.6390
      bal_acc      cnn       24.9951 0.5795 0.0021 0.5777 0.5823
      bal_

# Figure 4, Protocol C vs majority count

In [12]:
C = protocol_rows("protocolC").copy()

fig, axes = plt.subplots(1, len(HEADLINE), figsize=(4 * len(HEADLINE), 4))
all_rows = []
for ax, metric in zip(axes, HEADLINE):
    a = agg_metric(C, metric)
    for model in core.MODEL_NAMES:
        am = a[a["model"] == model].sort_values("n_maj")
        ax.errorbar(am["n_maj"], am["mean"], yerr=am["std"],
                    fmt=MODEL_STYLE[model], capsize=3, label=model)
        for _, rr in am.iterrows():
            all_rows.append({"metric": metric, "model": model, "n_maj": rr["n_maj"],
                             "mean": rr["mean"], "std": rr["std"],
                             "min": rr["min"], "max": rr["max"]})
    ax.set_xscale("log"); ax.set_xlabel("majority count n_maj"); ax.set_title(metric)
    if metric == HEADLINE[0]:
        ax.legend(fontsize=7)
fig.suptitle("Figure 4: Protocol C (fixed minority 1000) vs majority count")
_savefig(fig, "figure_04_protocolC_vs_nmaj")

dump = pd.DataFrame(all_rows)
dump.to_csv(os.path.join(FIGURES, "figure_04_protocolC_vs_nmaj.csv"), index=False)
_dump_df(dump, "Figure 4: Protocol C, metric vs majority count (mean/std/min/max)")
REPORT_NOTES.append(("protocolC",
    "Protocol C (minority fixed at 1000, growing majority): see figure_04 CSV for how "
    "minority metrics move as majority exposure increases at fixed minority availability."))

saved /kaggle/working/figures/figure_04_protocolC_vs_nmaj.png and .svg

=== Figure 4: Protocol C, metric vs majority count (mean/std/min/max) ===
       metric    model  n_maj   mean    std    min    max
      bal_acc logistic   1000 0.6029 0.0032 0.5994 0.6072
      bal_acc logistic   2000 0.5983 0.0023 0.5961 0.6015
      bal_acc logistic   5000 0.5817 0.0013 0.5798 0.5827
      bal_acc logistic  10000 0.5646 0.0035 0.5598 0.5681
      bal_acc logistic  25000 0.5466 0.0014 0.5448 0.5483
      bal_acc logistic  50000 0.5396 0.0011 0.5387 0.5411
      bal_acc logistic 100000 0.5358 0.0005 0.5354 0.5365
      bal_acc      cnn   1000 0.7841 0.0175 0.7615 0.8041
      bal_acc      cnn   2000 0.7598 0.0158 0.7411 0.7797
      bal_acc      cnn   5000 0.6964 0.0133 0.6809 0.7133
      bal_acc      cnn  10000 0.6397 0.0037 0.6367 0.6448
      bal_acc      cnn  25000 0.5777 0.0047 0.5719 0.5833
      bal_acc      cnn  50000 0.5428 0.0021 0.5408 0.5457
      bal_acc      cnn 100000 0.5206 0.003

# Figure 5, Protocol D vs minority count

In [13]:
D = protocol_rows("protocolD").copy()

fig, axes = plt.subplots(1, len(HEADLINE), figsize=(4 * len(HEADLINE), 4))
all_rows = []
for ax, metric in zip(axes, HEADLINE):
    a = agg_metric(D, metric)
    for model in core.MODEL_NAMES:
        am = a[a["model"] == model].sort_values("n_min")
        ax.errorbar(am["n_min"], am["mean"], yerr=am["std"],
                    fmt=MODEL_STYLE[model], capsize=3, label=model)
        for _, rr in am.iterrows():
            all_rows.append({"metric": metric, "model": model, "n_min": rr["n_min"],
                             "mean": rr["mean"], "std": rr["std"],
                             "min": rr["min"], "max": rr["max"]})
    ax.set_xscale("log"); ax.set_xlabel("minority count n_min"); ax.set_title(metric)
    if metric == HEADLINE[0]:
        ax.legend(fontsize=7)
fig.suptitle("Figure 5: Protocol D (fixed majority 32000) vs minority count")
_savefig(fig, "figure_05_protocolD_vs_nmin")

dump = pd.DataFrame(all_rows)
dump.to_csv(os.path.join(FIGURES, "figure_05_protocolD_vs_nmin.csv"), index=False)
_dump_df(dump, "Figure 5: Protocol D, metric vs minority count (mean/std/min/max)")
REPORT_NOTES.append(("protocolD",
    "Protocol D (majority fixed at 32000, removing minority): see figure_05 CSV for "
    "minority metrics as minority availability falls at fixed majority exposure."))

saved /kaggle/working/figures/figure_05_protocolD_vs_nmin.png and .svg

=== Figure 5: Protocol D, metric vs minority count (mean/std/min/max) ===
       metric    model  n_min   mean    std    min    max
      bal_acc logistic   1000 0.5456 0.0019 0.5431 0.5477
      bal_acc logistic   2000 0.5515 0.0007 0.5506 0.5521
      bal_acc logistic   4000 0.5636 0.0019 0.5610 0.5654
      bal_acc logistic   8000 0.5824 0.0022 0.5794 0.5845
      bal_acc logistic  16000 0.6184 0.0018 0.6161 0.6205
      bal_acc logistic  32000 0.6662 0.0023 0.6638 0.6693
      bal_acc      cnn   1000 0.5638 0.0011 0.5625 0.5650
      bal_acc      cnn   2000 0.6051 0.0039 0.5996 0.6081
      bal_acc      cnn   4000 0.6580 0.0128 0.6437 0.6748
      bal_acc      cnn   8000 0.7056 0.0085 0.6938 0.7129
      bal_acc      cnn  16000 0.7557 0.0094 0.7432 0.7660
      bal_acc      cnn  32000 0.7820 0.0044 0.7763 0.7869
      bal_acc resnet18   1000 0.6257 0.0018 0.6236 0.6281
      bal_acc resnet18   2000 0.6764 0.003

# Figure 6, decomposition (minority_ece and aurc_minority)

In [14]:
def decomposition_figure(metric, fig_no, stem):
    a_C = agg_metric(protocol_rows("protocolC"), metric).sort_values("n_maj")
    a_D = agg_metric(protocol_rows("protocolD"), metric).sort_values("n_min")
    a_B = agg_metric(protocol_rows("protocolB"), metric).sort_values("actual_ratio")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    all_rows = []
    for ax, (a, xcol, xlabel, panel) in zip(axes, [
            (a_C, "n_maj", "majority count (C)", "C"),
            (a_D, "n_min", "minority count (D)", "D"),
            (a_B, "actual_ratio", "actual ratio (B)", "B")]):
        for model in core.MODEL_NAMES:
            am = a[a["model"] == model].sort_values(xcol)
            ax.errorbar(am[xcol], am["mean"], yerr=am["std"],
                        fmt=MODEL_STYLE[model], capsize=3, label=model)
            for _, rr in am.iterrows():
                all_rows.append({"panel": panel, "metric": metric, "model": model,
                                 "x": rr[xcol], "mean": rr["mean"], "std": rr["std"]})
        ax.set_xscale("log"); ax.set_xlabel(xlabel); ax.set_title(f"panel {panel}: {metric}")
    axes[0].legend(fontsize=7)
    fig.suptitle(f"Figure {fig_no}: decomposition of {metric} (C, D, B)")
    _savefig(fig, stem)

    dump = pd.DataFrame(all_rows)
    dump.to_csv(os.path.join(FIGURES, stem + ".csv"), index=False)
    _dump_df(dump, f"Figure {fig_no}: decomposition of {metric}, all three panels")

decomposition_figure("minority_ece", 6, "figure_06_decomposition_minority_ece")
decomposition_figure("aurc_minority", 6, "figure_06b_decomposition_aurc_minority")

REPORT_NOTES.append(("decomposition",
    "Decomposition (Figure 6): Protocol C isolates changes associated with increasing "
    "majority exposure at fixed minority availability; Protocol D isolates changes "
    "associated with decreasing minority availability at fixed majority exposure; "
    "Protocol B shows both moving together. No numeric subtraction between protocols "
    "is performed. Values in figure_06 and figure_06b CSVs."))

saved /kaggle/working/figures/figure_06_decomposition_minority_ece.png and .svg

=== Figure 6: decomposition of minority_ece, all three panels ===
panel       metric    model           x   mean    std
    C minority_ece logistic   1000.0000 0.3025 0.0041
    C minority_ece logistic   2000.0000 0.3858 0.0040
    C minority_ece logistic   5000.0000 0.4761 0.0007
    C minority_ece logistic  10000.0000 0.5502 0.0092
    C minority_ece logistic  25000.0000 0.6258 0.0026
    C minority_ece logistic  50000.0000 0.6544 0.0018
    C minority_ece logistic 100000.0000 0.6657 0.0047
    C minority_ece      cnn   1000.0000 0.1632 0.0340
    C minority_ece      cnn   2000.0000 0.2444 0.0311
    C minority_ece      cnn   5000.0000 0.4011 0.0337
    C minority_ece      cnn  10000.0000 0.5447 0.0118
    C minority_ece      cnn  25000.0000 0.7077 0.0160
    C minority_ece      cnn  50000.0000 0.8133 0.0030
    C minority_ece      cnn 100000.0000 0.8861 0.0038
    C minority_ece resnet18   1000.0000 0.2

# Figure 7, model ladder

In [15]:
ladder_metric = "minority_ece"
Bc = agg_metric(protocol_rows("protocolB"), ladder_metric)

fig, ax = plt.subplots(figsize=(7, 5))
all_rows = []
for model in core.MODEL_NAMES:
    am = Bc[Bc["model"] == model].sort_values("actual_ratio")
    ax.errorbar(am["actual_ratio"], am["mean"], yerr=am["std"],
                fmt=MODEL_STYLE[model], capsize=3, label=model)
    for _, rr in am.iterrows():
        all_rows.append({"model": model, "actual_ratio": rr["actual_ratio"],
                         "mean": rr["mean"], "std": rr["std"]})
ax.set_xscale("log"); ax.set_xlabel("actual ratio"); ax.set_ylabel(ladder_metric)
ax.set_title("Figure 7: model ladder, minority_ece vs ratio (Protocol B)")
ax.legend()
_savefig(fig, "figure_07_model_ladder")

dump = pd.DataFrame(all_rows)
dump.to_csv(os.path.join(FIGURES, "figure_07_model_ladder.csv"), index=False)
_dump_df(dump, "Figure 7: model ladder, minority_ece vs ratio per model")
REPORT_NOTES.append(("ladder",
    "Model ladder (Figure 7): minority_ece vs ratio for logistic, cnn and resnet18 "
    "along Protocol B. Whether the residual net amplifies, attenuates or leaves the "
    "minority penalty unchanged is read from figure_07 CSV; three rungs is a coarse ladder."))

saved /kaggle/working/figures/figure_07_model_ladder.png and .svg

=== Figure 7: model ladder, minority_ece vs ratio per model ===
   model  actual_ratio   mean    std
logistic        1.0000 0.1388 0.0027
logistic        2.0000 0.3698 0.0024
logistic        4.9998 0.5360 0.0066
logistic       10.0003 0.5976 0.0021
logistic       24.9951 0.6375 0.0047
logistic       49.9960 0.6561 0.0064
logistic       99.9464 0.6642 0.0036
     cnn        1.0000 0.1477 0.0109
     cnn        2.0000 0.2334 0.0095
     cnn        4.9998 0.3961 0.0202
     cnn       10.0003 0.5362 0.0065
     cnn       24.9951 0.7001 0.0097
     cnn       49.9960 0.8038 0.0034
     cnn       99.9464 0.8816 0.0076
resnet18        1.0000 0.2572 0.0143
resnet18        2.0000 0.3104 0.0111
resnet18        4.9998 0.3789 0.0105
resnet18       10.0003 0.4591 0.0142
resnet18       24.9951 0.5587 0.0116
resnet18       49.9960 0.6912 0.0008
resnet18       99.9464 0.8140 0.0021


# Figure 8, reliability diagrams (ResNet-18, balanced vs extreme)

In [16]:
rel_targets = [("balanced (32000,32000)", 32000, 32000),
               ("extreme (634,63366) B r=100", 634, 63366)]
rel_lines = ["# Figure 8: per-class reliability bins (ResNet-18), three-seed mean", ""]
rel_csv_rows = []

fig, axes = plt.subplots(len(rel_targets), 2, figsize=(12, 5 * len(rel_targets)))
for ti, (label, nmin, nmaj) in enumerate(rel_targets):
    for ci, (cls_tag, cls_val) in enumerate([("minority (y==1)", 1), ("majority (y==0)", 0)]):
        ax = axes[ti][ci]
        per_seed_conf, per_seed_acc, per_seed_cnt = [], [], []
        for s in core.SEEDS:
            rid = core.make_run_id("resnet18", nmin, nmaj, s, s, s)
            test = _load_logits(rid, "test")
            if test is None:
                continue
            y, d = test
            p = metrics.probs_from_logit_diff(d)
            sub = y == cls_val
            lo, hi, cnt, conf, acc = metrics.reliability_bins(
                y[sub], p[sub], mode="top_label")
            per_seed_conf.append(conf); per_seed_acc.append(acc); per_seed_cnt.append(cnt)
            for i in range(len(lo)):
                rel_csv_rows.append({"condition": label, "class": cls_tag, "seed": s,
                                     "bin_lo": lo[i], "bin_hi": hi[i], "count": int(cnt[i]),
                                     "confidence": conf[i], "accuracy": acc[i],
                                     "gap": (conf[i] - acc[i])})
        if not per_seed_conf:
            ax.set_title(f"{label}\n{cls_tag}\n[missing]"); continue
        conf_m = np.nanmean(np.vstack(per_seed_conf), axis=0)
        acc_m = np.nanmean(np.vstack(per_seed_acc), axis=0)
        cnt_m = np.mean(np.vstack(per_seed_cnt), axis=0)
        edges = np.linspace(0, 1, metrics.ECE_BINS + 1)
        ax.plot(conf_m, acc_m, "o-", alpha=0.8)
        ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
        ax.set_xlabel("confidence"); ax.set_ylabel("accuracy")
        ax.set_title(f"{label}\n{cls_tag}")
        rel_lines.append(f"## {label}  {cls_tag}  (three-seed mean)")
        rel_lines.append("bin_lo | bin_hi | mean_count | mean_conf | mean_acc | gap")
        for i in range(metrics.ECE_BINS):
            gap = conf_m[i] - acc_m[i]
            rel_lines.append(f"{edges[i]:.3f} | {edges[i+1]:.3f} | {cnt_m[i]:.1f} | "
                             f"{conf_m[i]:.4f} | {acc_m[i]:.4f} | {gap:.4f}")
        rel_lines.append("")
fig.suptitle("Figure 8: per-class reliability, ResNet-18, three-seed mean")
_savefig(fig, "figure_08_reliability_resnet18")

pd.DataFrame(rel_csv_rows).to_csv(
    os.path.join(FIGURES, "figure_08_reliability_resnet18.csv"), index=False)
rel_path = os.path.join(SUMM, "07_reliability_bins.md")
with open(rel_path, "w") as f:
    f.write("\n".join(rel_lines) + "\n")
print("full per-bin tables (per-seed in CSV, three-seed mean in md) ->", rel_path)
print("[Figure 8] minority and majority bin tables differ; see", rel_path)
REPORT_NOTES.append(("reliability",
    "Reliability (Figure 8): per-class bins for ResNet-18 at balanced and extreme "
    "conditions, averaged over seeds {21,22,23}, with bin counts. Minority and majority "
    "tables differ. Per-seed values in figure_08 CSV, three-seed means in "
    "summaries/07_reliability_bins.md."))

/tmp/ipykernel_22/1183776241.py:29: RuntimeWarning: Mean of empty slice
  conf_m = np.nanmean(np.vstack(per_seed_conf), axis=0)
/tmp/ipykernel_22/1183776241.py:30: RuntimeWarning: Mean of empty slice
  acc_m = np.nanmean(np.vstack(per_seed_acc), axis=0)
/tmp/ipykernel_22/1183776241.py:29: RuntimeWarning: Mean of empty slice
  conf_m = np.nanmean(np.vstack(per_seed_conf), axis=0)
/tmp/ipykernel_22/1183776241.py:30: RuntimeWarning: Mean of empty slice
  acc_m = np.nanmean(np.vstack(per_seed_acc), axis=0)
/tmp/ipykernel_22/1183776241.py:29: RuntimeWarning: Mean of empty slice
  conf_m = np.nanmean(np.vstack(per_seed_conf), axis=0)
/tmp/ipykernel_22/1183776241.py:30: RuntimeWarning: Mean of empty slice
  acc_m = np.nanmean(np.vstack(per_seed_acc), axis=0)
/tmp/ipykernel_22/1183776241.py:29: RuntimeWarning: Mean of empty slice
  conf_m = np.nanmean(np.vstack(per_seed_conf), axis=0)
/tmp/ipykernel_22/1183776241.py:30: RuntimeWarning: Mean of empty slice
  acc_m = np.nanmean(np.vstack(per_see

saved /kaggle/working/figures/figure_08_reliability_resnet18.png and .svg
full per-bin tables (per-seed in CSV, three-seed mean in md) -> /kaggle/working/summaries/07_reliability_bins.md
[Figure 8] minority and majority bin tables differ; see /kaggle/working/summaries/07_reliability_bins.md


# Figure 9, risk-coverage (ResNet-18)

In [17]:
rc_targets = [("balanced (32000,32000)", 32000, 32000),
              ("extreme (634,63366)", 634, 63366)]
COMMON_COV = np.linspace(0.05, 1.0, 100)
FIXED_COV = [0.2, 0.4, 0.6, 0.8, 0.9, 0.95]
rc_csv_rows, rc_fixed_rows, rc_aurc_rows = [], [], []

fig, axes = plt.subplots(1, len(rc_targets), figsize=(6 * len(rc_targets), 5))
for ax, (label, nmin, nmaj) in zip(axes, rc_targets):
    for name, cls in [("overall", None), ("minority", 1), ("majority", 0)]:
        seed_curves, seed_aurc = [], []
        for s in core.SEEDS:
            rid = core.make_run_id("resnet18", nmin, nmaj, s, s, s)
            test = _load_logits(rid, "test")
            if test is None:
                continue
            y, d = test
            p = metrics.probs_from_logit_diff(d)
            mask = None if cls is None else (y == cls)
            cov, risk, aurc = metrics.risk_coverage(y, p, subset_mask=mask)
            seed_curves.append(np.interp(COMMON_COV, cov, risk))
            seed_aurc.append(aurc)
        if not seed_curves:
            continue
        curves = np.vstack(seed_curves)
        risk_mean = curves.mean(axis=0)
        ax.plot(COMMON_COV, risk_mean,
                label=f"{name} (AURC={np.mean(seed_aurc):.4f})", alpha=0.85)
        for i in range(0, len(COMMON_COV), 10):
            rc_csv_rows.append({"condition": label, "subset": name,
                                "coverage": COMMON_COV[i],
                                "risk_mean": risk_mean[i],
                                "risk_std": curves[:, i].std(ddof=0)})
        for fc in FIXED_COV:
            j = int(np.argmin(np.abs(COMMON_COV - fc)))
            rc_fixed_rows.append({"condition": label, "subset": name, "coverage": fc,
                                  "risk_mean": float(curves[:, j].mean()),
                                  "risk_std": float(curves[:, j].std(ddof=0))})
        rc_aurc_rows.append({"condition": label, "subset": name,
                             "aurc_mean": float(np.mean(seed_aurc)),
                             "aurc_std": float(np.std(seed_aurc, ddof=0))})
    ax.set_xlabel("coverage"); ax.set_ylabel("selective risk"); ax.set_title(label)
    ax.legend(fontsize=7)
fig.suptitle("Figure 9: risk-coverage, ResNet-18, three-seed mean")
_savefig(fig, "figure_09_risk_coverage_resnet18")

pd.DataFrame(rc_csv_rows).to_csv(
    os.path.join(FIGURES, "figure_09_risk_coverage_resnet18.csv"), index=False)
_dump_df(pd.DataFrame(rc_fixed_rows), "Figure 9: risk at fixed coverage (mean/std over 3 seeds)")
_dump_df(pd.DataFrame(rc_aurc_rows), "Figure 9: AURC overall/minority/majority (mean/std over 3 seeds)")
REPORT_NOTES.append(("riskcoverage",
    "Risk-coverage (Figure 9): overall, minority and majority curves for ResNet-18, "
    "three-seed mean, with risk at fixed coverage and AURC mean/std in the dumps. At full "
    "coverage the minority-subset risk equals 1 - recall_pos by construction, so near-1 "
    "minority AURC at high skew is partly arithmetic and is not independent evidence."))

saved /kaggle/working/figures/figure_09_risk_coverage_resnet18.png and .svg

=== Figure 9: risk at fixed coverage (mean/std over 3 seeds) ===
             condition   subset  coverage  risk_mean  risk_std
balanced (32000,32000)  overall      0.20     0.0389    0.0111
balanced (32000,32000)  overall      0.40     0.0771    0.0208
balanced (32000,32000)  overall      0.60     0.1103    0.0167
balanced (32000,32000)  overall      0.80     0.1538    0.0085
balanced (32000,32000)  overall      0.90     0.1807    0.0052
balanced (32000,32000)  overall      0.95     0.1946    0.0039
balanced (32000,32000) minority      0.20     0.1358    0.0432
balanced (32000,32000) minority      0.40     0.2412    0.0462
balanced (32000,32000) minority      0.60     0.2905    0.0277
balanced (32000,32000) minority      0.80     0.3350    0.0145
balanced (32000,32000) minority      0.90     0.3520    0.0114
balanced (32000,32000) minority      0.95     0.3611    0.0086
balanced (32000,32000) majority      0.

# Figure 10, confidence and entropy correct vs incorrect

In [18]:
ce_targets = [("balanced", 32000, 32000), ("extreme", 634, 63366)]
split_keys = [
    "mean_conf_correct_min", "mean_conf_incorrect_min",
    "mean_conf_correct_maj", "mean_conf_incorrect_maj",
    "mean_entropy_correct_min", "mean_entropy_incorrect_min",
    "mean_entropy_correct_maj", "mean_entropy_incorrect_maj",
]
rows_dump = []
for label, nmin, nmaj in ce_targets:
    cid = core.make_condition_id(nmin, nmaj)
    g = master[(master["condition_id"] == cid) & (master["model"] == "resnet18") &
               (master["data_seed"].isin(MAIN_SEEDS)) &
               (~master["protocol"].astype(str).str.contains("init_sensitivity"))]
    for k in split_keys + ["delta_conf"]:
        v = pd.to_numeric(g[k], errors="coerce").to_numpy()
        v = v[np.isfinite(v)]
        rows_dump.append({"condition": label, "metric": k,
                          "mean": float(v.mean()) if len(v) else float("nan"),
                          "std": float(v.std(ddof=0)) if len(v) else float("nan")})

dump = pd.DataFrame(rows_dump)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
conf_keys = split_keys[:4]
ent_keys = split_keys[4:]
for ax, keys, title in [(axes[0], conf_keys, "confidence"), (axes[1], ent_keys, "entropy (bits)")]:
    xs = np.arange(len(ce_targets)); width = 0.2
    for i, k in enumerate(keys):
        means = [dump[(dump["condition"] == lbl) & (dump["metric"] == k)]["mean"].iloc[0]
                 for lbl, _, _ in ce_targets]
        errs = [dump[(dump["condition"] == lbl) & (dump["metric"] == k)]["std"].iloc[0]
                for lbl, _, _ in ce_targets]
        ax.bar(xs + i * width, means, width, yerr=errs, capsize=3, label=k)
    ax.set_xticks(xs + 1.5 * width); ax.set_xticklabels([l for l, _, _ in ce_targets])
    ax.set_title(title); ax.legend(fontsize=6)
fig.suptitle("Figure 10: class-split confidence and entropy, correct vs incorrect (ResNet-18)")
_savefig(fig, "figure_10_conf_entropy")

dump.to_csv(os.path.join(FIGURES, "figure_10_conf_entropy.csv"), index=False)
_dump_df(dump, "Figure 10: every bar (mean/std over 3 seeds), minority vs majority, ResNet-18")
REPORT_NOTES.append(("confentropy",
    "Confidence and entropy (Figure 10): split by true class (minority y==1, majority "
    "y==0) and correctness, three-seed mean and std, balanced vs extreme, ResNet-18. "
    "Classes are not pooled. Values in figure_10 CSV."))

saved /kaggle/working/figures/figure_10_conf_entropy.png and .svg

=== Figure 10: every bar (mean/std over 3 seeds), minority vs majority, ResNet-18 ===
condition                     metric   mean    std
 balanced      mean_conf_correct_min 0.9092 0.0043
 balanced    mean_conf_incorrect_min 0.8539 0.0190
 balanced      mean_conf_correct_maj 0.9566 0.0015
 balanced    mean_conf_incorrect_maj 0.7545 0.0026
 balanced   mean_entropy_correct_min 0.3137 0.0140
 balanced mean_entropy_incorrect_min 0.4728 0.0528
 balanced   mean_entropy_correct_maj 0.1809 0.0062
 balanced mean_entropy_incorrect_maj 0.6940 0.0074
 balanced                 delta_conf 0.0955 0.0153
  extreme      mean_conf_correct_min 0.7320 0.0012
  extreme    mean_conf_incorrect_min 0.9307 0.0006
  extreme      mean_conf_correct_maj 0.9941 0.0003
  extreme    mean_conf_incorrect_maj 0.6782 0.0341
  extreme   mean_entropy_correct_min 0.7559 0.0027
  extreme mean_entropy_incorrect_min 0.2658 0.0004
  extreme   mean_entropy_correc

# Figure 11, recalibration diagnostic (three states)

In [19]:
state_metrics = {"minority_ece": ("minority_ece", "minority_ece_prior", "minority_ece_ts"),
                 "classwise_ece": ("classwise_ece", "classwise_ece_prior", "classwise_ece_ts")}
recal_models = core.MODEL_NAMES

fig, axes = plt.subplots(len(recal_models), len(state_metrics),
                         figsize=(7 * len(state_metrics), 4.5 * len(recal_models)))
all_rows = []
for ri, model in enumerate(recal_models):
    Bm = protocol_rows("protocolB")
    Bm = Bm[Bm["model"] == model]
    for cj, (mname, (raw_c, pr_c, ts_c)) in enumerate(state_metrics.items()):
        ax = axes[ri][cj]
        for state_label, col in [("raw", raw_c), ("prior", pr_c), ("TS", ts_c)]:
            a = agg_metric(Bm, col).sort_values("actual_ratio")
            ax.errorbar(a["actual_ratio"], a["mean"], yerr=a["std"],
                        fmt="o-", capsize=3, label=state_label)
            for _, rr in a.iterrows():
                all_rows.append({"model": model, "metric": mname, "state": state_label,
                                 "actual_ratio": rr["actual_ratio"],
                                 "mean": rr["mean"], "std": rr["std"]})
        ax.set_xscale("log"); ax.set_xlabel("actual ratio")
        ax.set_title(f"{model}: {mname}"); ax.legend(fontsize=7)
fig.suptitle("Figure 11: recalibration states (raw, prior, TS) across ratios, all models")
_savefig(fig, "figure_11_recalibration_states")

dump = pd.DataFrame(all_rows)
dump.to_csv(os.path.join(FIGURES, "figure_11_recalibration_states.csv"), index=False)
_dump_df(dump, "Figure 11: minority_ece and classwise_ece in raw/prior/TS, all three models")

# recall_pos vs recall_pos_prior along Protocol B: the class-conditional test of
# whether prior correction restores minority detection. Report, do not conclude.
rp_rows = []
for model in core.MODEL_NAMES:
    Bm = protocol_rows("protocolB")
    Bm = Bm[Bm["model"] == model]
    for col in ("recall_pos", "recall_pos_prior"):
        a = agg_metric(Bm, col).sort_values("actual_ratio")
        for _, rr in a.iterrows():
            rp_rows.append({"model": model, "metric": col, "actual_ratio": rr["actual_ratio"],
                            "mean": rr["mean"], "std": rr["std"]})
rp = pd.DataFrame(rp_rows)
rp.to_csv(os.path.join(FIGURES, "figure_11b_recall_prior.csv"), index=False)
_dump_df(rp, "Figure 11: recall_pos vs recall_pos_prior across ratios (mean/std over 3 seeds)")
REPORT_NOTES.append(("recalibration",
    "Recalibration (Figure 11): minority_ece and classwise_ece in raw, prior and TS states "
    "across Protocol B ratios, for all three models; and recall_pos beside recall_pos_prior "
    "in figure_11b CSV. Numbers reported; no conclusion drawn here about whether any remedy "
    "removes the class-conditional gap."))

saved /kaggle/working/figures/figure_11_recalibration_states.png and .svg

=== Figure 11: minority_ece and classwise_ece in raw/prior/TS, all three models ===
   model        metric state  actual_ratio   mean    std
logistic  minority_ece   raw        1.0000 0.1388 0.0027
logistic  minority_ece   raw        2.0000 0.3698 0.0024
logistic  minority_ece   raw        4.9998 0.5360 0.0066
logistic  minority_ece   raw       10.0003 0.5976 0.0021
logistic  minority_ece   raw       24.9951 0.6375 0.0047
logistic  minority_ece   raw       49.9960 0.6561 0.0064
logistic  minority_ece   raw       99.9464 0.6642 0.0036
logistic  minority_ece prior        1.0000 0.1388 0.0027
logistic  minority_ece prior        2.0000 0.1426 0.0018
logistic  minority_ece prior        4.9998 0.0186 0.0019
logistic  minority_ece prior       10.0003 0.1374 0.0029
logistic  minority_ece prior       24.9951 0.1295 0.0008
logistic  minority_ece prior       49.9960 0.0792 0.0002
logistic  minority_ece prior       99.9464 

# Figure 12, ISIC with bootstrap CIs

In [20]:
isic_rows = []
isic_master = []
if isic_ledger_path and os.path.exists(isic_ledger_path):
    IL = pd.read_csv(isic_ledger_path)
    IL = (IL.assign(_r=IL["status"].map(order).fillna(-1))
            .sort_values("_r").drop_duplicates("run_id", keep="last").drop(columns="_r"))
    isic_succ = IL[IL["status"] == "success"]
    print(f"ISIC ledger: {len(isic_succ)} success rows")

    for _, r in isic_succ.iterrows():
        rid = r["run_id"]
        path = os.path.join(logits_dir, f"{rid}_holdout.csv")
        if not os.path.exists(path):
            print("  missing ISIC holdout logits:", rid); continue
        df = pd.read_csv(path)
        y = df["y_true"].to_numpy().astype(int)
        d = df["logit_diff"].to_numpy().astype(np.float64)
        n_min, n_maj = int(r["n_min"]), int(r["n_maj"])
        pi_train = n_min / (n_min + n_maj)
        # ISIC has no separate val split; TS and temperature are in-sample and
        # excluded from reporting below. The raw and prior metrics plus the
        # bootstrap CIs are the valid ISIC outputs.
        m = metrics.compute_all_metrics(y, d, y, d, pi_train, with_bootstrap=True)
        row = {"run_id": rid, "model": r["model"], "n_min": n_min, "n_maj": n_maj,
               "target_ratio": r["target_ratio"], "actual_ratio": r["actual_ratio"],
               "data_seed": r["data_seed"], "pi_train": pi_train}
        row.update(m)
        isic_master.append(row)

    im = pd.DataFrame(isic_master)
    im.to_csv(os.path.join(RESULTS, "isic_master_results.csv"), index=False)
    print("wrote results/isic_master_results.csv, shape", im.shape)
    print("note: isic *_ts and temperature columns are written but invalid "
          "(TS fit in-sample on the holdout); excluded from the figure and dump")

    prim = ["recall_pos", "auroc", "brier", "aurc_minority"]
    fig, axes = plt.subplots(1, len(prim), figsize=(5 * len(prim), 4))
    for ax, metric in zip(axes, prim):
        for model in core.MODEL_NAMES:
            gm = im[im["model"] == model]
            agg = gm.groupby("n_maj").agg(
                mean=(metric, "mean"),
                lo=(metric + "_lo", "mean"),
                hi=(metric + "_hi", "mean")).reset_index().sort_values("n_maj")
            yerr = np.vstack([agg["mean"] - agg["lo"], agg["hi"] - agg["mean"]])
            ax.errorbar(agg["n_maj"], agg["mean"], yerr=yerr,
                        fmt=MODEL_STYLE[model], capsize=3, label=model)
            for _, rr in agg.iterrows():
                isic_rows.append({"metric": metric, "model": model, "n_maj": rr["n_maj"],
                                  "mean": rr["mean"], "lo": rr["lo"], "hi": rr["hi"]})
        ax.set_xscale("log"); ax.set_xlabel("benign majority count"); ax.set_title(metric)
        if metric == prim[0]:
            ax.legend(fontsize=7)
    fig.suptitle("Figure 12: ISIC external check, primary metrics with bootstrap CIs")
    _savefig(fig, "figure_12_isic")

    dump = pd.DataFrame(isic_rows)
    dump.to_csv(os.path.join(FIGURES, "figure_12_isic.csv"), index=False)
    _dump_df(dump, "Figure 12: ISIC point estimates with bootstrap lo/hi (TS/temperature excluded)")
    REPORT_NOTES.append(("isic",
        "ISIC external check (Figure 12): recall_pos, auroc, brier and aurc_minority with "
        "stratified percentile bootstrap CIs. lo/hi are per-run bootstrap bounds averaged "
        "over the three seeds, not an interval on the three-seed mean. ISIC temperature "
        "scaling is excluded (fit in-sample on the holdout). The claim is that the direction "
        "of the minority-reliability effect is consistent in a second domain, not that it "
        "generalises across medical imaging."))
else:
    print("ISIC ledger not found; ISIC figure skipped")
    REPORT_NOTES.append(("isic", "[PENDING: ISIC ledger not found at aggregation]"))

ISIC ledger: 45 success rows
wrote results/isic_master_results.csv, shape (45, 57)
note: isic *_ts and temperature columns are written but invalid (TS fit in-sample on the holdout); excluded from the figure and dump
saved /kaggle/working/figures/figure_12_isic.png and .svg

=== Figure 12: ISIC point estimates with bootstrap lo/hi (TS/temperature excluded) ===
       metric    model   n_maj   mean     lo     hi
   recall_pos logistic   384.0 0.7301 0.6649 0.7917
   recall_pos logistic  1920.0 0.2645 0.1993 0.3297
   recall_pos logistic  3840.0 0.1341 0.0870 0.1866
   recall_pos logistic  9600.0 0.0417 0.0163 0.0707
   recall_pos logistic 19200.0 0.0254 0.0054 0.0507
   recall_pos      cnn   384.0 0.7808 0.7174 0.8370
   recall_pos      cnn  1920.0 0.3732 0.3025 0.4438
   recall_pos      cnn  3840.0 0.1522 0.1014 0.2029
   recall_pos      cnn  9600.0 0.0380 0.0163 0.0652
   recall_pos      cnn 19200.0 0.0109 0.0000 0.0272
   recall_pos resnet18   384.0 0.7826 0.7228 0.8406
   recall_pos 

# Figure 13, init-sensitivity vs data-seed spread

In [21]:
init_metric = "minority_ece"
INIT_SEEDS = [21, 22, 23, 24, 25]
compare = []

for model in ["logistic", "cnn"]:
    for (nmin, nmaj) in [(1000, 1000), (2000, 2000)]:
        cid = core.make_condition_id(nmin, nmaj)
        cond = (master["model"] == model) & (master["condition_id"] == cid)

        # init-only: data=loader=21 fixed, init_seed varies 21-25
        ir = master[cond & (master["data_seed"] == 21) & (master["loader_seed"] == 21) &
                    (master["init_seed"].isin(INIT_SEEDS))].drop_duplicates("run_id")
        iv = pd.to_numeric(ir[init_metric], errors="coerce").to_numpy()
        iv = iv[np.isfinite(iv)]
        init_only_std = float(iv.std(ddof=0)) if len(iv) else float("nan")

        # main three-seed: all three streams tied, seeds 21-23 (seed-21 row
        # kept regardless of its dual protocol tag)
        mr = master[cond & (master["data_seed"].isin(MAIN_SEEDS)) &
                    (master["init_seed"] == master["data_seed"]) &
                    (master["loader_seed"] == master["data_seed"])].drop_duplicates("run_id")
        mv = pd.to_numeric(mr[init_metric], errors="coerce").to_numpy()
        mv = mv[np.isfinite(mv)]
        three_seed_std = float(mv.std(ddof=0)) if len(mv) else float("nan")

        compare.append({"model": model, "condition": cid,
                        "init_only_std": init_only_std, "n_init": len(iv),
                        "three_seed_std": three_seed_std, "n_main": len(mv)})

dump = pd.DataFrame(compare)
if (dump["n_main"] != 3).any() or (dump["n_init"] != 5).any():
    print("WARNING: expected n_main=3 and n_init=5 in every cell; check seed selection")
    print(dump[(dump["n_main"] != 3) | (dump["n_init"] != 5)].to_string(index=False))
else:
    print("seed counts correct: n_main=3 and n_init=5 in every cell")

fig, ax = plt.subplots(figsize=(9, 5))
xs = np.arange(len(dump)); width = 0.35
ax.bar(xs - width / 2, dump["init_only_std"], width, label="init-only spread (5 init seeds)")
ax.bar(xs + width / 2, dump["three_seed_std"], width,
       label="three-seed spread (data+init+loader tied)")
ax.set_xticks(xs)
ax.set_xticklabels([f"{r['model']}\n{r['condition']}" for _, r in dump.iterrows()], fontsize=7)
ax.set_ylabel(f"std of {init_metric}")
ax.set_title("Figure 13: initialisation-only spread vs tied three-seed spread")
ax.legend(fontsize=8)
_savefig(fig, "figure_13_init_sensitivity")

dump.to_csv(os.path.join(FIGURES, "figure_13_init_sensitivity.csv"), index=False)
_dump_df(dump, "Figure 13: init-only std (5 inits) vs three-seed std (tied), with run counts")
REPORT_NOTES.append(("init",
    "Initialisation sensitivity (Figure 13): the five-init-seed spread of minority_ece "
    "(data and loader seeds fixed at 21) is initialisation-only variability; the three-seed "
    "spread (seeds 21, 22, 23, with data, initialisation and loader order tied) bundles all "
    "three sources and is not pure data-sampling variance. Values and run counts in "
    "figure_13 CSV."))

seed counts correct: n_main=3 and n_init=5 in every cell
saved /kaggle/working/figures/figure_13_init_sensitivity.png and .svg

=== Figure 13: init-only std (5 inits) vs three-seed std (tied), with run counts ===
   model   condition  init_only_std  n_init  three_seed_std  n_main
logistic m1000_M1000         0.0015       5          0.0041       3
logistic m2000_M2000         0.0007       5          0.0101       3
     cnn m1000_M1000         0.0156       5          0.0340       3
     cnn m2000_M2000         0.0149       5          0.0072       3


# Figure 14 (appendix), training curves and exposures

In [22]:
curve_files = sorted(glob.glob(os.path.join(curves_dir, "*.csv"))) if curves_dir else []
sample = []
for model in core.MODEL_NAMES:
    for f in curve_files:
        if os.path.basename(f).startswith(model + "__"):
            sample.append(f); break

fig, ax = plt.subplots(figsize=(8, 5))
for f in sample:
    cv = pd.read_csv(f)
    ax.plot(cv["step"], cv["train_loss"], label=os.path.basename(f).replace(".csv", ""),
            alpha=0.8)
ax.set_xlabel("step"); ax.set_ylabel("train loss")
ax.set_title("Figure 14: representative training curves")
ax.legend(fontsize=6)
_savefig(fig, "figure_14_training_curves")

exp = master[["protocol", "model", "condition_id", "n_min", "n_maj",
              "exposures_per_image", "effective_epochs", "actual_steps"]].copy()
exp = exp.drop_duplicates(["model", "condition_id"]).sort_values(["model", "n_min", "n_maj"])
exp_path = os.path.join(SUMM, "07_exposures.md")
with open(exp_path, "w") as f:
    f.write("# Figure 14 backing: exposures and effective epochs per condition\n\n")
    f.write(exp.round(2).to_markdown(index=False))
    f.write("\n")
exp.to_csv(os.path.join(FIGURES, "figure_14_training_curves.csv"), index=False)
print("[Figure 14] exposures table written to", exp_path)
print(exp.round(2).to_string(index=False))
REPORT_NOTES.append(("appendix",
    "Appendix (Figure 14): representative training curves and the full exposures-per-image "
    "and effective-epochs table in summaries/07_exposures.md."))

saved /kaggle/working/figures/figure_14_training_curves.png and .svg
[Figure 14] exposures table written to /kaggle/working/summaries/07_exposures.md
                     protocol    model    condition_id  n_min  n_maj  exposures_per_image  effective_epochs  actual_steps
                    protocolB      cnn     m634_M63366    634  63366                10.00             10.00          5000
   init_sensitivity|protocolC      cnn     m1000_M1000   1000   1000               320.00            320.00          5000
                    protocolC      cnn     m1000_M2000   1000   2000               213.33            213.33          5000
                    protocolC      cnn     m1000_M5000   1000   5000               106.67            106.67          5000
                    protocolC      cnn    m1000_M10000   1000  10000                58.18             58.18          5000
                    protocolC      cnn    m1000_M25000   1000  25000                24.62             24.62          5

# assemble final_report.md

In [23]:
notes = dict(REPORT_NOTES)

def _num(df, col, cond):
    sub = df[cond]
    v = pd.to_numeric(sub[col], errors="coerce").to_numpy()
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return "[PENDING: no value]"
    return f"{v.mean():.4f} (sd {v.std(ddof=0):.4f}, n={len(v)})"

lines = []
w = lines.append
w("# DATAwise: final results report")
w("")
w(f"Config hash: {core.CONFIG_HASH}. All numbers are read from "
  "results/master_results.csv and the figure backing CSVs.")
w("")
w("## Dataset and design")
w("- PCam balanced eval splits: valid 32,738, test 32,754, each exactly 50/50, "
  "trimmed once with EVAL_BALANCE_SEED.")
w(f"- Models: {', '.join(core.MODEL_NAMES)}. Main seeds: {core.SEEDS} (data=init=loader "
  f"tied). Step budget {core.STEP_BUDGET}, batch {core.BATCH_SIZE}, fp32, from scratch.")
w(f"- PCam main successful runs aggregated: "
  f"{int((~master['protocol'].astype(str).str.contains('init_sensitivity')).sum())}; "
  f"init-sensitivity runs held separate: "
  f"{int(master['protocol'].astype(str).str.contains('init_sensitivity').sum())}.")
w("")
w("## Per-protocol findings (observational)")
for key in ("protocolA", "protocolB", "protocolC", "protocolD"):
    w(f"- {notes.get(key, '[PENDING: ' + key + ']')}")
w("")
w("## Class-conditional calibration headline (Protocol B extreme, resnet18)")
_ext = ((master["condition_id"] == core.make_condition_id(634, 63366)) &
        (master["model"] == "resnet18") & (master["data_seed"].isin(MAIN_SEEDS)))
w(f"- minority_ece (raw): {_num(master, 'minority_ece', _ext)}")
w(f"- minority_ece (prior): {_num(master, 'minority_ece_prior', _ext)}")
w(f"- minority_ece (TS): {_num(master, 'minority_ece_ts', _ext)}")
w(f"- majority_ece (raw): {_num(master, 'majority_ece', _ext)}")
w(f"- recall_pos (raw): {_num(master, 'recall_pos', _ext)}")
w(f"- recall_pos (prior-corrected): {_num(master, 'recall_pos_prior', _ext)}")
w("  Numbers reported; no conclusion drawn here on whether a remedy removes the gap.")
w("")
w("## Decomposition")
w(notes.get("decomposition", "[PENDING: decomposition]"))
w("")
w("## Model ladder")
w(notes.get("ladder", "[PENDING: ladder]"))
w("")
w("## Recalibration diagnostic")
w(notes.get("recalibration", "[PENDING: recalibration]"))
w("")
w("## ISIC external check")
w(notes.get("isic", "[PENDING: isic]"))
w("")
w("## Initialisation sensitivity")
w(notes.get("init", "[PENDING: init]"))
w("")
w("## Consistency and reproducibility")
w(notes.get("consistency", "[PENDING: consistency]"))
w(f"- Single config_hash across all runs: {core.CONFIG_HASH}.")
w("- Saved artefacts: class permutations, subset indices, balanced-eval indices, per-run "
  "test and validation logits, training curves, frozen config.")
w("- Metric computation is isolated in metrics.py, a deliberate and documented departure "
  "from the original blueprint (which placed metrics in core.py), so the frozen run engine "
  "was never edited after the runs began. No scientific definition changed.")
w("- Evaluation splits trimmed to exact 50/50 once at load by a fixed seeded rule, kept "
  "indices saved; every reported metric is per-class, rank-based or a per-sample proper "
  "score and is robust to that trim.")
w("")
w("## Data quality")
w("- See summaries/07_data_quality.md for the calibration-collapse check (ece vs "
  "minority_ece and classwise_ece), NaN/inf, temperature-bound flags, seed coverage and "
  "hash uniqueness.")

report_path = os.path.join(SUMM, "final_report.md")
with open(report_path, "w") as f:
    f.write("\n".join(lines) + "\n")
print("wrote", report_path)
print("\n".join(lines))

wrote /kaggle/working/summaries/final_report.md
# DATAwise: final results report

Config hash: 03fa342b535795c1. All numbers are read from results/master_results.csv and the figure backing CSVs.

## Dataset and design
- PCam balanced eval splits: valid 32,738, test 32,754, each exactly 50/50, trimmed once with EVAL_BALANCE_SEED.
- Models: logistic, cnn, resnet18. Main seeds: [21, 22, 23] (data=init=loader tied). Step budget 5000, batch 128, fp32, from scratch.
- PCam main successful runs aggregated: 221; init-sensitivity runs held separate: 20.

## Per-protocol findings (observational)
- Protocol A (balanced, varying total N): see figure_02 CSV for per-model trends in bal_acc, auroc, brier, minority_ece and aurc_minority as total N grows.
- Protocol B (fixed total N_F=64000, varying ratio): the r=100 condition sits at actual ratio ~99.95 by the pinned count formula. See figure_03 CSV.
- Protocol C (minority fixed at 1000, growing majority): see figure_04 CSV for how minority metrics mo

# persistence

In [24]:
if core.DATAWISE_RESULTS_SLUG is not None:
    core.push_results_store()
    print("pushed results store to", core.DATAWISE_RESULTS_SLUG)
else:
    print("DATAWISE_RESULTS_SLUG is unset; persistence skipped this session.")
    print("WARNING: without it, master_results.csv and the figures will not survive session end.")

Starting upload for file isic.zip


100%|██████████| 490k/490k [00:00<00:00, 1.75MB/s]


Upload successful: isic.zip (490KB)
Starting upload for file summaries.zip


100%|██████████| 7.40k/7.40k [00:00<00:00, 28.8kB/s]


Upload successful: summaries.zip (7KB)
Starting upload for file condition_registry.csv


...

Upload successful: frozen_config.md (431B)
Starting upload for file training_curves.zip


100%|██████████| 116k/116k [00:00<00:00, 395kB/s]


Upload successful: training_curves.zip (116KB)
Starting upload for file logs.zip


100%|██████████| 61.9k/61.9k [00:00<00:00, 220kB/s]


Upload successful: logs.zip (62KB)
pushed results store to hrushisanap/datawise-results
